In [1]:
#foodRecSys-1

import pandas as pd
import numpy as np
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os as os

/Users/makabaka/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

dataset_dir = "/Users/makabaka/.cache/kagglehub/datasets/elisaxxygao/foodrecsysv1/versions/1"

raw_recipes = pd.read_csv(os.path.join(dataset_dir, "raw-data_recipe.csv"))
core_recipes = pd.read_csv(os.path.join(dataset_dir, "core-data_recipe.csv"))

raw_interactions = pd.read_csv(os.path.join(dataset_dir, "raw-data_interaction.csv"))

train_df = pd.read_csv(os.path.join(dataset_dir, "core-data-train_rating.csv"))
valid_df = pd.read_csv(os.path.join(dataset_dir, "core-data-valid_rating.csv"))
test_df  = pd.read_csv(os.path.join(dataset_dir, "core-data-test_rating.csv"))


#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_colwidth', None)


print("=== Recipe Data ===")
print("Raw recipes:", raw_recipes.shape)
print("Core recipes:", core_recipes.shape)

print("\n=== Interaction Data ===")
print("Raw interactions:", raw_interactions.shape)
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

print("\n=== Columns Preview ===")
print("Raw recipes columns:", raw_recipes.columns.tolist())
print("Interactions columns:", raw_interactions.columns.tolist())

print("\n=== Sample Raw Recipes ===")
display(raw_recipes.head())

print("\n=== Sample Train Interactions ===")
display(train_df.head())

=== Recipe Data ===
Raw recipes: (49698, 9)
Core recipes: (45630, 6)

=== Interaction Data ===
Raw interactions: (3794003, 4)
Train: (676946, 4)
Valid: (133459, 4)
Test : (283440, 4)

=== Columns Preview ===
Raw recipes columns: ['recipe_id', 'recipe_name', 'aver_rate', 'image_url', 'review_nums', 'ingredients', 'cooking_directions', 'nutritions', 'reviews']
Interactions columns: ['user_id', 'recipe_id', 'rating', 'dateLastModified']

=== Sample Raw Recipes ===


,recipe_id,recipe_name,aver_rate,image_url,review_nums,ingredients,cooking_directions,nutritions,reviews
0,222388,Homemade Bacon,5.000000,https://images.media-allrecipes.com/userphotos...,3,pork belly^smoked paprika^kosher salt^ground b...,{'directions': u'Prep\n5 m\nCook\n2 h 45 m\nRe...,"{u'niacin': {u'hasCompleteData': False, u'name...","{8542392: {'rating': 5, 'followersCount': 11, ..."
1,240488,"Pork Loin, Apples, and Sauerkraut",4.764706,https://images.media-allrecipes.com/userphotos...,29,sauerkraut drained^Granny Smith apples sliced^...,{'directions': u'Prep\n15 m\nCook\n2 h 30 m\nR...,"{u'niacin': {u'hasCompleteData': False, u'name...","{3574785: {'rating': 5, 'followersCount': 0, '..."
2,218939,Foolproof Rosemary Chicken Wings,4.571429,https://images.media-allrecipes.com/userphotos...,12,chicken wings^sprigs rosemary^head garlic^oliv...,"{'directions': u""Prep\n20 m\nCook\n40 m\nReady...","{u'niacin': {u'hasCompleteData': True, u'name'...","{13774946: {'rating': 5, 'followersCount': 0, ..."
3,87211,Chicken Pesto Paninis,4.625000,https://images.media-allrecipes.com/userphotos...,163,focaccia bread quartered^prepared basil pesto^...,{'directions': u'Prep\n15 m\nCook\n5 m\nReady ...,"{u'niacin': {u'hasCompleteData': True, u'name'...","{1563136: {'rating': 5, 'followersCount': 0, '..."
4,245714,Potato Bacon Pizza,4.500000,https://images.media-allrecipes.com/userphotos...,2,red potatoes^strips bacon^Sauce:^heavy whippin...,{'directions': u'Prep\n20 m\nCook\n45 m\nReady...,"{u'niacin': {u'hasCompleteData': True, u'name'...","{2945555: {'rating': 5, 'followersCount': 6690..."



=== Sample Train Interactions ===


,user_id,recipe_id,rating,dateLastModified
0,5215572,17991,5,2010-08-25T14:38:53.84\n
1,5215572,170724,4,2010-09-09T14:04:45.733\n
2,5215572,18045,5,2010-08-16T14:51:25.833\n
3,3622615,60598,4,2009-03-15T12:10:20.85\n
4,1313770,47519,5,2005-10-04T15:43:36.653\n


In [3]:
import pandas as pd
import ast
import re

recipes_v1 = raw_recipes.copy()

#parse nutrition safely
def safe_parse(x):
    if pd.isna(x):
        return None
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None

    # remove python2 unicode prefix
    x = re.sub(r"\bu'", "'", x)
    x = re.sub(r'\bu"', '"', x)

    try:
        return ast.literal_eval(x)
    except:
        return None

recipes_v1["nutritions_parsed"] = recipes_v1["nutritions"].apply(safe_parse)

#flatten nutrition → columns
def flatten_nutrition(nutrition_dict):
    if not isinstance(nutrition_dict, dict):
        return {}

    flat = {}
    for key, value in nutrition_dict.items():
        if isinstance(value, dict):
            flat[key] = value.get("amount")
    return flat

nutrition_df = recipes_v1["nutritions_parsed"] \
    .apply(flatten_nutrition) \
    .apply(pd.Series)

#convert all to numeric
nutrition_df = nutrition_df.apply(pd.to_numeric, errors="coerce")

#merge back to main dataframe
recipes_v1 = pd.concat([recipes_v1, nutrition_df], axis=1)

print("Created nutrition columns:", nutrition_df.columns.tolist())
display(recipes_v1.head())

Created nutrition columns: ['niacin', 'sugars', 'sodium', 'carbohydrates', 'vitaminB6', 'calories', 'thiamin', 'fat', 'folate', 'caloriesFromFat', 'calcium', 'fiber', 'magnesium', 'iron', 'cholesterol', 'protein', 'vitaminA', 'potassium', 'saturatedFat', 'vitaminC']


,recipe_id,recipe_name,aver_rate,image_url,review_nums,ingredients,cooking_directions,nutritions,reviews,nutritions_parsed,...,calcium,fiber,magnesium,iron,cholesterol,protein,vitaminA,potassium,saturatedFat,vitaminC
0,222388,Homemade Bacon,5.000000,https://images.media-allrecipes.com/userphotos...,3,pork belly^smoked paprika^kosher salt^ground b...,{'directions': u'Prep\n5 m\nCook\n2 h 45 m\nRe...,"{u'niacin': {u'hasCompleteData': False, u'name...","{8542392: {'rating': 5, 'followersCount': 11, ...","{'niacin': {'hasCompleteData': False, 'name': ...",...,11.18365,0.531887,21.65558,1.240848,61.73750,21.002540,474.20730,347.2267,7.736815,0.776127
1,240488,"Pork Loin, Apples, and Sauerkraut",4.764706,https://images.media-allrecipes.com/userphotos...,29,sauerkraut drained^Granny Smith apples sliced^...,{'directions': u'Prep\n15 m\nCook\n2 h 30 m\nR...,"{u'niacin': {u'hasCompleteData': False, u'name...","{3574785: {'rating': 5, 'followersCount': 0, '...","{'niacin': {'hasCompleteData': False, 'name': ...",...,135.45380,10.223360,80.73712,6.622245,99.20000,36.398780,73.17785,1088.9230,3.646474,52.768480
2,218939,Foolproof Rosemary Chicken Wings,4.571429,https://images.media-allrecipes.com/userphotos...,12,chicken wings^sprigs rosemary^head garlic^oliv...,"{'directions': u""Prep\n20 m\nCook\n40 m\nReady...","{u'niacin': {u'hasCompleteData': True, u'name'...","{13774946: {'rating': 5, 'followersCount': 0, ...","{'niacin': {'hasCompleteData': True, 'name': '...",...,60.08832,0.886681,24.21298,1.704567,71.40000,23.912650,359.36400,249.1827,5.683611,5.307448
3,87211,Chicken Pesto Paninis,4.625000,https://images.media-allrecipes.com/userphotos...,163,focaccia bread quartered^prepared basil pesto^...,{'directions': u'Prep\n15 m\nCook\n5 m\nReady ...,"{u'niacin': {u'hasCompleteData': True, u'name'...","{1563136: {'rating': 5, 'followersCount': 0, '...","{'niacin': {'hasCompleteData': True, 'name': '...",...,528.46170,4.441335,64.67662,5.011620,61.38938,32.375370,604.75370,348.8067,10.918760,18.015020
4,245714,Potato Bacon Pizza,4.500000,https://images.media-allrecipes.com/userphotos...,2,red potatoes^strips bacon^Sauce:^heavy whippin...,{'directions': u'Prep\n20 m\nCook\n45 m\nReady...,"{u'niacin': {u'hasCompleteData': True, u'name'...","{2945555: {'rating': 5, 'followersCount': 6690...","{'niacin': {'hasCompleteData': True, 'name': '...",...,132.22650,0.749092,11.41930,1.024803,21.55610,7.059566,168.32450,106.8477,3.975159,0.905797


In [4]:
import pandas as pd
import ast
import re

recsys_df = recipes_v1.copy()

# -----------------------------
# 1. parse cooking_directions to get plain directions text
# -----------------------------
def safe_parse(x):
    if pd.isna(x):
        return None
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None

    x = re.sub(r"\bu'", "'", x)
    x = re.sub(r'\bu"', '"', x)

    try:
        return ast.literal_eval(x)
    except Exception:
        return None

def extract_directions(x):
    parsed = safe_parse(x)
    if isinstance(parsed, dict):
        return parsed.get("directions")
    return None

recsys_df["directions"] = recsys_df["cooking_directions"].apply(extract_directions)

# -----------------------------
# 2. rename columns to Food.com-style schema
# -----------------------------
recsys_df = recsys_df.rename(columns={
    "recipe_name": "recipe_title",
    "aver_rate": "rating_value",
    "review_nums": "review_count",
    "fat": "total_fat_amount",
    "saturatedFat": "saturated_fat_amount",
    "cholesterol": "cholesterol_amount",
    "sodium": "sodium_amount",
    "carbohydrates": "total_carbohydrate_amount",
    "fiber": "dietary_fiber_amount",
    "sugars": "total_sugars_amount",
    "protein": "protein_amount"
})

# -----------------------------
# 3. keep only aligned columns
# -----------------------------
foodcom_columns = [
    "recipe_id",
    "recipe_title",
    "ingredients",
    "directions",
    "image_url",
    "rating_value",
    "review_count",
    "calories",
    "total_fat_amount",
    "saturated_fat_amount",
    "cholesterol_amount",
    "sodium_amount",
    "total_carbohydrate_amount",
    "dietary_fiber_amount",
    "total_sugars_amount",
    "protein_amount"
]

recsys_aligned = recsys_df[foodcom_columns].copy()
recsys_aligned["source"] = "foodrecsys"

print(recsys_aligned.shape)
print(recsys_aligned.columns.tolist())
pd.set_option('display.max_colwidth', None)

display(recsys_aligned.head(2))

(49698, 17)
['recipe_id', 'recipe_title', 'ingredients', 'directions', 'image_url', 'rating_value', 'review_count', 'calories', 'total_fat_amount', 'saturated_fat_amount', 'cholesterol_amount', 'sodium_amount', 'total_carbohydrate_amount', 'dietary_fiber_amount', 'total_sugars_amount', 'protein_amount', 'source']


,recipe_id,recipe_title,ingredients,directions,image_url,rating_value,review_count,calories,total_fat_amount,saturated_fat_amount,cholesterol_amount,sodium_amount,total_carbohydrate_amount,dietary_fiber_amount,total_sugars_amount,protein_amount,source
0,222388,Homemade Bacon,pork belly^smoked paprika^kosher salt^ground black pepper,"Prep\n5 m\nCook\n2 h 45 m\nReady In\n11 h 50 m\nPreheat oven to 200 degrees F (95 degrees C).\nSeason pork belly with paprika, salt, and pepper. Tightly wrap pork twice in heavy-duty aluminum foil. Place on a baking sheet and bake in the preheated oven for 2 1/2 hours. Turn off the oven; let pork rest in the oven for 1 hour. Remove meat from oven, leaving it wrapped in aluminum foil, and refrigerate at least 8 hours or overnight.\nRemove pork from foil and slice across the grain in 1/4-inch thick slices. Working in batches, cook pork in a non-stick skillet over medium heat until golden and crisped, 6 to 8 minutes per slice.",https://images.media-allrecipes.com/userphotos/720x405/876328.jpg,5.000000,3,308.1481,23.58587,7.736815,61.7375,2017.130,1.797819,0.531887,0.093559,21.00254,foodrecsys
1,240488,"Pork Loin, Apples, and Sauerkraut",sauerkraut drained^Granny Smith apples sliced^large onion^caraway seeds^apple cider divided^brown sugar^Rub:^Thai seasoning^salt^garlic powder^ground black pepper^boneless pork loin roast,"Prep\n15 m\nCook\n2 h 30 m\nReady In\n2 h 45 m\nPreheat oven to 325 degrees F (165 degrees C).\nMix sauerkraut, apples, onion, and caraway seeds in a large roasting pan. Stir 1/4 cup apple cider and brown sugar together in a separate bowl; pour over sauerkraut mixture.\nStir Thai seasoning, salt, garlic powder, and black pepper together in a small bowl; rub onto the top and bottom of the roast.\nMake an indentation in the center of the sauerkraut mixture and place the seasoned roast in the indentation. Pour the remaining apple cider around the roast.\nBake in the preheated oven for 1 hour; baste roast with juices. Continue baking roast, basting every 30 minutes, until cooked through, 2 1/2 to 3 hours. An instant-read thermometer inserted into the center should read at least 145 degrees F (63 degrees C).",https://images.media-allrecipes.com/userphotos/720x405/1897772.jpg,4.764706,29,371.7219,11.67521,3.646474,99.2000,2606.764,32.081760,10.223360,19.841460,36.39878,foodrecsys


In [ ]:
import pandas as pd
import re

def extract_times_and_directions(text):
    if pd.isna(text) or not isinstance(text, str):
        return pd.Series(
            [None, None, None, None, None, None, None],
            index=[
                "prep_time",
                "cook_time",
                "total_time",
                "directions_clean",
                "PrepTime_Minutes",
                "CookTime_Minutes",
                "TotalTime_Minutes"
            ]
        )

    text = text.replace("\\n", "\n")

    prep_match = re.search(r'Prep\s*\n([^\n]+)', text)
    cook_match = re.search(r'Cook\s*\n([^\n]+)', text)
    total_match = re.search(r'Ready In\s*\n([^\n]+)', text)

    prep_time = prep_match.group(1).strip() if prep_match else None
    cook_time = cook_match.group(1).strip() if cook_match else None
    total_time = total_match.group(1).strip() if total_match else None

    directions = re.sub(r'Prep\s*\n[^\n]+\n', '', text)
    directions = re.sub(r'Cook\s*\n[^\n]+\n', '', directions)
    directions = re.sub(r'Ready In\s*\n[^\n]+\n', '', directions)
    directions = directions.strip()

    def to_minutes(t):
        if pd.isna(t) or not isinstance(t, str):
            return None
        h = re.search(r'(\d+)\s*h', t)
        m = re.search(r'(\d+)\s*m', t)
        hours = int(h.group(1)) if h else 0
        mins = int(m.group(1)) if m else 0
        return hours * 60 + mins

    return pd.Series(
        [
            prep_time,
            cook_time,
            total_time,
            directions,
            to_minutes(prep_time),
            to_minutes(cook_time),
            to_minutes(total_time)
        ],
        index=[
            "prep_time",
            "cook_time",
            "total_time",
            "directions_clean",
            "PrepTime_Minutes",
            "CookTime_Minutes",
            "TotalTime_Minutes"
        ]
    )


,prep_time,cook_time,total_time,PrepTime_Minutes,CookTime_Minutes,TotalTime_Minutes,directions
0,5 m,2 h 45 m,11 h 50 m,5.0,165.0,710.0,"Preheat oven to 200 degrees F (95 degrees C).\nSeason pork belly with paprika, salt, and pepper. Tightly wrap pork twice in heavy-duty aluminum foil. Place on a baking sheet and bake in the preheated oven for 2 1/2 hours. Turn off the oven; let pork rest in the oven for 1 hour. Remove meat from oven, leaving it wrapped in aluminum foil, and refrigerate at least 8 hours or overnight.\nRemove pork from foil and slice across the grain in 1/4-inch thick slices. Working in batches, cook pork in a non-stick skillet over medium heat until golden and crisped, 6 to 8 minutes per slice."
1,15 m,2 h 30 m,2 h 45 m,15.0,150.0,165.0,"Preheat oven to 325 degrees F (165 degrees C).\nMix sauerkraut, apples, onion, and caraway seeds in a large roasting pan. Stir 1/4 cup apple cider and brown sugar together in a separate bowl; pour over sauerkraut mixture.\nStir Thai seasoning, salt, garlic powder, and black pepper together in a small bowl; rub onto the top and bottom of the roast.\nMake an indentation in the center of the sauerkraut mixture and place the seasoned roast in the indentation. Pour the remaining apple cider around the roast.\nBake in the preheated oven for 1 hour; baste roast with juices. Continue baking roast, basting every 30 minutes, until cooked through, 2 1/2 to 3 hours. An instant-read thermometer inserted into the center should read at least 145 degrees F (63 degrees C)."


In [ ]:
# apply to the current directions column
time_cols = recsys_aligned["directions"].apply(extract_times_and_directions)

# join back
recsys_aligned = pd.concat([recsys_aligned, time_cols], axis=1)

# replace old directions with cleaned directions
recsys_aligned["directions"] = recsys_aligned["directions_clean"]
recsys_aligned = recsys_aligned.drop(columns=["directions_clean"])

display(
    recsys_aligned[
        [
            "prep_time",
            "cook_time",
            "total_time",
            "PrepTime_Minutes",
            "CookTime_Minutes",
            "TotalTime_Minutes",
            "directions"
        ]
    ].head(2)
)

In [6]:
def convert_ingredients(text):
    if not isinstance(text, str):
        return []
    
    return [i.strip().lower() for i in text.split("^") if i.strip()]


In [7]:
recsys_aligned["ingredients_list"] = recsys_aligned["ingredients"].apply(convert_ingredients)

In [8]:
recsys_aligned.head(1)

,recipe_id,recipe_title,ingredients,directions,image_url,rating_value,review_count,calories,total_fat_amount,saturated_fat_amount,...,total_sugars_amount,protein_amount,source,prep_time,cook_time,total_time,PrepTime_Minutes,CookTime_Minutes,TotalTime_Minutes,ingredients_list
0,222388,Homemade Bacon,pork belly^smoked paprika^kosher salt^ground black pepper,"Preheat oven to 200 degrees F (95 degrees C).\nSeason pork belly with paprika, salt, and pepper. Tightly wrap pork twice in heavy-duty aluminum foil. Place on a baking sheet and bake in the preheated oven for 2 1/2 hours. Turn off the oven; let pork rest in the oven for 1 hour. Remove meat from oven, leaving it wrapped in aluminum foil, and refrigerate at least 8 hours or overnight.\nRemove pork from foil and slice across the grain in 1/4-inch thick slices. Working in batches, cook pork in a non-stick skillet over medium heat until golden and crisped, 6 to 8 minutes per slice.",https://images.media-allrecipes.com/userphotos/720x405/876328.jpg,5.0,3,308.1481,23.58587,7.736815,...,0.093559,21.00254,foodrecsys,5 m,2 h 45 m,11 h 50 m,5.0,165.0,710.0,"[pork belly, smoked paprika, kosher salt, ground black pepper]"
